# LLM GMC — Name That LLM baseline

This competition-linked notebook computes a reproducible local validation score. It does not submit predictions to the leaderboard.

Baseline: word + character TF-IDF features with class-balanced LinearSVC. If `prompt_id` is present, validation is grouped by prompt to reduce prompt leakage.

In [ ]:
from pathlib import Path
import json
import time
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

DATA_DIR = Path('/kaggle/input/competitions/name-that-llm')
print('Dataset directory exists:', DATA_DIR.exists())
print('Dataset files:', sorted(p.name for p in DATA_DIR.glob('*')) if DATA_DIR.exists() else [])
assert (DATA_DIR / 'train.csv').exists(), f'train.csv not found under {DATA_DIR}'
train = pd.read_csv(DATA_DIR / 'train.csv')
print('Train shape:', train.shape)
print('Columns:', train.columns.tolist())
assert {'text', 'model'}.issubset(train.columns), 'Expected text and model columns'
train['text'] = train['text'].fillna('').astype(str)
train['model'] = train['model'].astype(str)
print('Model classes:', train['model'].nunique())
display(train['model'].value_counts())

In [ ]:
if 'prompt_id' in train.columns and train['prompt_id'].notna().any():
    groups = train['prompt_id'].fillna('__missing_prompt_id__').astype(str)
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
    train_idx, valid_idx = next(splitter.split(train, groups=groups))
    split_method = 'GroupShuffleSplit by prompt_id'
else:
    train_idx, valid_idx = train_test_split(range(len(train)), test_size=0.20, random_state=42, stratify=train['model'])
    split_method = 'stratified random split (prompt_id unavailable)'

train_part = train.iloc[list(train_idx)].copy()
valid_part = train.iloc[list(valid_idx)].copy()
print('Validation method:', split_method)
print('Training rows:', len(train_part))
print('Validation rows:', len(valid_part))
print('Validation classes:', valid_part['model'].nunique())

In [ ]:
import re
import numpy as np
from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# This diagnostic keeps every prompt group inside one fold.
groups = train['prompt_id'].fillna('__missing_prompt_id__').astype(str)
texts = train['text'].fillna('').astype(str).tolist()
labels = train['model'].to_numpy()

def stylometric_features(items):
    rows = []
    function_words = ['the', 'a', 'an', 'and', 'but', 'or', 'if', 'to', 'of', 'in', 'on', 'for', 'with', 'as', 'is', 'are', 'was', 'be', 'that', 'this', 'it', 'I', 'we', 'you', 'can', 'would', 'should']
    punctuation = '.,;:!?()[]{}\"\'/-'
    for text in items:
        words = re.findall(r"\b[\w']+\b", text.lower())
        word_lengths = np.asarray([len(w) for w in words], dtype=float)
        n_words = max(len(words), 1)
        n_chars = max(len(text), 1)
        sentences = max(len(re.findall(r'[.!?]+', text)), 1)
        punct_counts = [text.count(ch) / n_chars for ch in punctuation]
        function_counts = [words.count(w) / n_words for w in function_words]
        rows.append([
            np.log1p(len(text)), np.log1p(len(words)), np.log1p(sentences),
            np.mean(word_lengths) if len(word_lengths) else 0.0,
            np.std(word_lengths) if len(word_lengths) else 0.0,
            len(set(words)) / n_words,
            sum(ch.isdigit() for ch in text) / n_chars,
            sum(ch.isupper() for ch in text) / n_chars,
            sum(ch.isspace() for ch in text) / n_chars,
            text.count('\\n') / n_chars, text.count('\\n\\n') / n_chars,
            *punct_counts,
            text.count('#') / n_chars, text.count('*') / n_chars,
            text.count('```') / n_chars,
            *function_counts,
        ])
    return np.asarray(rows, dtype=np.float32)

style_matrix = stylometric_features(texts)

def make_text_features():
    return FeatureUnion([
        ('word', TfidfVectorizer(
            analyzer='word', ngram_range=(1, 3), min_df=2,
            sublinear_tf=True, max_features=220000)),
        ('char', TfidfVectorizer(
            analyzer='char', ngram_range=(2, 6), min_df=2,
            sublinear_tf=True, max_features=260000)),
    ])

fold_results = []
gkf = GroupKFold(n_splits=5)
for fold, (tr_idx, va_idx) in enumerate(gkf.split(texts, labels, groups=groups), 1):
    train_text = [texts[i] for i in tr_idx]
    valid_text = [texts[i] for i in va_idx]
    y_tr, y_va = labels[tr_idx], labels[va_idx]

    text_features = make_text_features()
    X_tr_text = text_features.fit_transform(train_text)
    X_va_text = text_features.transform(valid_text)
    text_model = LinearSVC(C=1.0, class_weight='balanced')
    text_model.fit(X_tr_text, y_tr)
    text_pred = text_model.predict(X_va_text)

    scaler = StandardScaler()
    X_tr_style = scaler.fit_transform(style_matrix[tr_idx])
    X_va_style = scaler.transform(style_matrix[va_idx])
    style_model = LogisticRegression(C=1.0, max_iter=2500, class_weight='balanced')
    style_model.fit(X_tr_style, y_tr)
    style_pred = style_model.predict(X_va_style)

    X_tr_combined = hstack([X_tr_text, csr_matrix(X_tr_style)], format='csr')
    X_va_combined = hstack([X_va_text, csr_matrix(X_va_style)], format='csr')
    combined_model = LinearSVC(C=1.0, class_weight='balanced')
    combined_model.fit(X_tr_combined, y_tr)
    combined_pred = combined_model.predict(X_va_combined)

    row = {
        'fold': fold,
        'text_accuracy': float(accuracy_score(y_va, text_pred)),
        'text_balanced_accuracy': float(balanced_accuracy_score(y_va, text_pred)),
        'style_accuracy': float(accuracy_score(y_va, style_pred)),
        'style_balanced_accuracy': float(balanced_accuracy_score(y_va, style_pred)),
        'combined_accuracy': float(accuracy_score(y_va, combined_pred)),
        'combined_balanced_accuracy': float(balanced_accuracy_score(y_va, combined_pred)),
        'validation_rows': int(len(va_idx)),
    }
    fold_results.append(row)
    print(
        f"Fold {fold}: text={row['text_balanced_accuracy']:.4f}, "
        f"style={row['style_balanced_accuracy']:.4f}, "
        f"combined={row['combined_balanced_accuracy']:.4f}"
    )

summary = {
    'experiment': 'LLM GMC five-fold grouped fingerprint diagnostic',
    'competition': 'name-that-llm',
    'split_method': '5-fold GroupKFold by prompt_id',
    'total_rows': int(len(train)),
    'model_classes': int(train['model'].nunique()),
    'fold_results': fold_results,
}
for key in ['text_balanced_accuracy', 'style_balanced_accuracy', 'combined_balanced_accuracy']:
    values = [r[key] for r in fold_results]
    summary[key + '_mean'] = float(np.mean(values))
    summary[key + '_std'] = float(np.std(values, ddof=1))
    summary[key + '_min'] = float(np.min(values))
    summary[key + '_max'] = float(np.max(values))
print('\\nSUMMARY:', json.dumps(summary, indent=2))
Path('/kaggle/working/diagnostic_results.json').write_text(json.dumps(summary, indent=2))
print('Saved diagnostic results:', '/kaggle/working/diagnostic_results.json')


## Diagnostic interpretation

This run compares lexical TF-IDF, explicit stylometric fingerprint features, and their combination under 5-fold prompt-grouped validation. The score is local validation, not the Kaggle leaderboard score.
